# DEP Experimentation

### Part 1 — Feature Selection on Structured Data

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
model = LogisticRegression(max_iter=10000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Baseline Accuracy:", accuracy_score(y_test, y_pred))

Baseline Accuracy: 0.956140350877193


In [5]:
from sklearn.feature_selection import SelectKBest, chi2

selector = SelectKBest(chi2, k=10)

X_new = selector.fit_transform(X, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_new, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Chi-square Accuracy:", accuracy_score(y_test, pred))

Chi-square Accuracy: 0.9649122807017544


In [6]:
from sklearn.feature_selection import RFE

selector = RFE(model, n_features_to_select=10)

X_rfe = selector.fit_transform(X, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_rfe, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("RFE Accuracy:", accuracy_score(y_test, pred))

RFE Accuracy: 0.9736842105263158


In [7]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, pred))

Random Forest Accuracy: 0.956140350877193


### Part 2 — Text Feature Extraction



In [8]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

In [9]:
texts = [
    "Win money now",
    "Hello how are you",
    "Free prize claim now",
    "Lets meet tomorrow",
    "Congratulations you won"
]

labels = [1,0,1,0,1]

In [10]:
vectorizer = CountVectorizer()

X = vectorizer.fit_transform(texts)

model = MultinomialNB()

model.fit(X, labels)

pred = model.predict(X)

print("BOW Accuracy:", accuracy_score(labels, pred))

BOW Accuracy: 1.0


In [11]:
tfidf = TfidfVectorizer()

X = tfidf.fit_transform(texts)

model.fit(X, labels)

pred = model.predict(X)

print("TF-IDF Accuracy:", accuracy_score(labels, pred))

TF-IDF Accuracy: 1.0


### Part 3 — Image Feature Extraction

In [12]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.svm import SVC

In [13]:
digits = load_digits()

X = digits.data
y = digits.target

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = SVC()

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy without PCA:", accuracy_score(y_test, pred))

Accuracy without PCA: 0.9861111111111112


In [15]:
pca = PCA(n_components=30)

X_pca = pca.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy with PCA:", accuracy_score(y_test, pred))

Accuracy with PCA: 0.9861111111111112


### Part 4 — Time Series Feature Extraction

In [16]:
data = {
    "Passengers":[112,118,132,129,121,135,148,148,136,119]
}

df = pd.DataFrame(data)

In [17]:
df["lag1"] = df["Passengers"].shift(1)
df["lag2"] = df["Passengers"].shift(2)

In [18]:
df["rolling_mean"] = df["Passengers"].rolling(window=3).mean()
df["rolling_std"] = df["Passengers"].rolling(window=3).std()

df = df.dropna()

print(df)

   Passengers   lag1   lag2  rolling_mean  rolling_std
2         132  118.0  112.0    120.666667    10.263203
3         129  132.0  118.0    126.333333     7.371115
4         121  129.0  132.0    127.333333     5.686241
5         135  121.0  129.0    128.333333     7.023769
6         148  135.0  121.0    134.666667    13.503086
7         148  148.0  135.0    143.666667     7.505553
8         136  148.0  148.0    144.000000     6.928203
9         119  136.0  148.0    134.333333    14.571662


In [19]:
from sklearn.linear_model import LinearRegression

X = df[["lag1","lag2","rolling_mean","rolling_std"]]
y = df["Passengers"]

model = LinearRegression()

model.fit(X,y)

pred = model.predict(X)

print("Predictions:", pred)

Predictions: [132. 129. 121. 135. 148. 148. 136. 119.]
